# Intermediate 02 lab — From visual evidence to checked conclusions

This lab builds a small, transparent multimodal reasoning system for an industrial maintenance question:

> Does the scratched valve require maintenance under the rule **scratch present AND valve left of pipe AND fewer than three bolts**?

The notebook exposes only reviewable work products—task nodes, observations, evidence bindings, deterministic tool events, checks, and the terminal decision. It does not request, imitate, or store hidden chain-of-thought.

![A bounded question moves through observable artifacts to a checked conclusion.](assets/checked-reasoning-pipeline.svg)


## 1. Reproducibility, source separation, and risk boundary

- `Factory A` supplies construction examples.
- `Factory B` is development-only and may select the review threshold.
- `Factory C` is held-out test reporting only. It never selects a threshold, tool tolerance, prompt, or rule.
- Related counterfactuals are created inside a split after source assignment.
- `local_structured_perception_proxy` reads a procedural scene record. It is **not a VLM, foundation model, or production perception benchmark**.
- Outputs are advisory and carry `authorization = "none"`.


In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import random
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

SEED = 20260906
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 80)

FACT_STATES = {"verified_true", "verified_false", "uncertain", "unknown"}
DECISION_STATES = {"decision_true", "decision_false", "review_required"}
SOURCE_CONTRACT = {
    "construction": "Factory A",
    "development": "Factory B",
    "test": "Factory C",
}
LOCAL_PERCEPTION_LABEL = "local_structured_perception_proxy"
DEMONSTRATION_THRESHOLD_NOTICE = "Demonstration thresholds for this notebook runtime only."

print({
    "seed": SEED,
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": Image.__version__,
    "matplotlib": matplotlib.__version__,
})


## 2. Generate source-separated visual scenes

The visual surface changes by factory, while the causal rule remains fixed. Color is deliberately irrelevant so we can detect a language/appearance shortcut. Some panels have low-contrast scratches, creating a review case without changing the hidden ground truth.


In [ ]:
IMAGE_SIZE = 96
FACTORY_STYLES = {
    "Factory A": {"background": (236, 239, 244), "panel": (218, 224, 231), "pipe": (80, 89, 103)},
    "Factory B": {"background": (244, 237, 224), "panel": (226, 216, 199), "pipe": (91, 80, 69)},
    "Factory C": {"background": (218, 232, 229), "panel": (195, 214, 210), "pipe": (58, 86, 84)},
}


def make_record(factory: str, index: int, split: str) -> dict[str, Any]:
    rng = np.random.default_rng(SEED + 97 * index + sum(map(ord, factory)))
    scratch = bool((index % 3) != 0)
    side = "left" if (index % 4) < 2 else "right"
    bolt_count = int(1 + (index % 4))
    color = "red" if bool(rng.integers(0, 2)) else "blue"
    low_contrast = bool(index % 7 == 0)
    return {
        "scene_id": f"{factory.lower().replace(' ', '-')}-{index:03d}",
        "group_id": f"{factory.lower().replace(' ', '-')}-group-{index:03d}",
        "factory": factory,
        "split": split,
        "scratch": scratch,
        "side": side,
        "bolt_count": bolt_count,
        "color": color,
        "low_contrast": low_contrast,
        "decision_truth": bool(scratch and side == "left" and bolt_count < 3),
    }


records = []
for split, factory, size in [
    ("construction", "Factory A", 8),
    ("development", "Factory B", 28),
    ("test", "Factory C", 28),
]:
    records.extend(make_record(factory, i, split) for i in range(size))

records_df = pd.DataFrame(records)
assert set(records_df.groupby("group_id")["split"].nunique()) == {1}
assert set(records_df.groupby("factory")["split"].nunique()) == {1}
records_df.groupby(["split", "factory"])[["scratch", "decision_truth"]].agg(["count", "mean"])


In [ ]:
def scene_geometry(record: dict[str, Any]) -> dict[str, tuple[int, int, int, int]]:
    valve = (14, 32, 38, 56) if record["side"] == "left" else (62, 32, 86, 56)
    pipe = (46, 13, 54, 82)
    vx1, vy1, vx2, vy2 = valve
    bolt_centers = [(vx1 + 5, vy1 + 5), (vx2 - 5, vy1 + 5), (vx1 + 5, vy2 - 5), (vx2 - 5, vy2 - 5)]
    geometry = {"valve_1": valve, "pipe_1": pipe}
    for i, (cx, cy) in enumerate(bolt_centers[: record["bolt_count"]], start=1):
        geometry[f"bolt_{i}"] = (cx - 2, cy - 2, cx + 2, cy + 2)
    return geometry


def render_scene(record: dict[str, Any]) -> Image.Image:
    style = FACTORY_STYLES[record["factory"]]
    image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), style["background"])
    draw = ImageDraw.Draw(image)
    draw.rounded_rectangle((5, 7, 91, 89), radius=5, fill=style["panel"], outline=(90, 100, 112), width=2)
    geometry = scene_geometry(record)
    draw.rounded_rectangle(geometry["pipe_1"], radius=3, fill=style["pipe"])
    valve_color = (190, 58, 63) if record["color"] == "red" else (55, 101, 180)
    draw.ellipse(geometry["valve_1"], fill=valve_color, outline=(30, 35, 42), width=2)
    for object_id, box in geometry.items():
        if object_id.startswith("bolt_"):
            draw.ellipse(box, fill=(235, 191, 74), outline=(55, 51, 44))
    if record["scratch"]:
        x1, y1, x2, y2 = geometry["valve_1"]
        scratch_color = tuple(int((a + b) / 2) for a, b in zip(valve_color, style["panel"])) if record["low_contrast"] else (245, 245, 238)
        draw.line((x1 + 5, y2 - 5, x2 - 5, y1 + 5), fill=scratch_color, width=2)
    return image


sample_rows = records_df.groupby("split", sort=False).head(2).to_dict(orient="records")
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, record in zip(axes.flat, sample_rows):
    ax.imshow(render_scene(record))
    ax.set_title(f"{record['factory']} · {record['scene_id'].split('-')[-1]}\n"
                 f"scratch={record['scratch']}, side={record['side']}, bolts={record['bolt_count']}")
    ax.axis("off")
plt.tight_layout()


## 3. Explicit observations, claims, and tool events

The evidence object records image, object, and region identity. A claim stores its public state and provenance. These structures are an audit interface—not an explanation of private model computation.

![Evidence requirements differ by task capability.](assets/evidence-contracts.svg)


In [ ]:
@dataclass(frozen=True)
class Evidence:
    evidence_id: str
    image_id: str
    object_id: str
    region_xyxy: tuple[float, float, float, float]
    source: str = LOCAL_PERCEPTION_LABEL
    foundation_model: bool = False


@dataclass
class Claim:
    claim_id: str
    kind: str
    value: Any
    state: str
    evidence_ids: tuple[str, ...] = ()
    object_ids: tuple[str, ...] = ()
    image_ids: tuple[str, ...] = ()
    tool_event_id: str | None = None
    note: str = ""

    def __post_init__(self):
        if self.state not in FACT_STATES:
            raise ValueError(f"Unsupported fact state: {self.state}")


@dataclass(frozen=True)
class ToolContract:
    name: str
    version: str
    deterministic: bool
    input_schema: str
    output_schema: str


@dataclass
class ToolEvent:
    tool_event_id: str
    name: str
    version: str
    inputs: dict[str, Any]
    outputs: dict[str, Any]
    status: str
    latency_ms: float
    error: str | None = None


TOOL_CONTRACTS = {
    "spatial_relation": ToolContract("spatial_relation", "1.0.0", True, "subject_box/reference_box xyxy; 96px frame", "relation booleans + normalized distance"),
    "count_objects": ToolContract("count_objects", "1.0.0", True, "list[str] approved object IDs", "unique count + sorted IDs"),
    "less_than": ToolContract("less_than", "1.0.0", True, "numeric value and threshold", "boolean predicate"),
    "contradiction_check": ToolContract("contradiction_check", "1.0.0", True, "relation claims", "contradiction pairs"),
}


def centers(box):
    x1, y1, x2, y2 = map(float, box)
    if not (x2 > x1 and y2 > y1):
        raise ValueError("xyxy box must have positive area")
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def spatial_relation(subject_box, reference_box, image_size=IMAGE_SIZE, tolerance=1.0):
    sx, sy = centers(subject_box); rx, ry = centers(reference_box)
    return {
        "left_of": bool(sx < rx - tolerance),
        "right_of": bool(sx > rx + tolerance),
        "above": bool(sy < ry - tolerance),
        "below": bool(sy > ry + tolerance),
        "normalized_distance": float(math.dist((sx, sy), (rx, ry)) / (math.sqrt(2) * image_size)),
    }


def count_objects(object_ids):
    if not isinstance(object_ids, (list, tuple)) or not all(isinstance(v, str) for v in object_ids):
        raise TypeError("object_ids must be a list/tuple of strings")
    unique_ids = sorted(set(object_ids))
    return {"count": len(unique_ids), "object_ids": unique_ids}


def less_than(value, threshold):
    if isinstance(value, bool) or isinstance(threshold, bool):
        raise TypeError("boolean values are not numeric measurements")
    return {"result": bool(float(value) < float(threshold)), "value": float(value), "threshold": float(threshold)}


def run_tool(name: str, inputs: dict[str, Any], event_index: int) -> ToolEvent:
    contract = TOOL_CONTRACTS[name]
    started = time.perf_counter_ns()
    try:
        if name == "spatial_relation":
            outputs = spatial_relation(**inputs)
        elif name == "count_objects":
            outputs = count_objects(**inputs)
        elif name == "less_than":
            outputs = less_than(**inputs)
        else:
            raise KeyError(f"Tool is not allow-listed: {name}")
        status, error = "ok", None
    except Exception as exc:
        outputs, status, error = {}, "error", f"{type(exc).__name__}: {exc}"
    latency_ms = (time.perf_counter_ns() - started) / 1e6
    return ToolEvent(f"tool-{event_index:03d}", name, contract.version, inputs, outputs, status, latency_ms, error)


# Assertion-backed unit checks for the deterministic boundary.
assert spatial_relation((0, 0, 10, 10), (20, 0, 30, 10))["left_of"]
assert spatial_relation((20, 0, 30, 10), (0, 0, 10, 10))["right_of"]
assert count_objects(["bolt_2", "bolt_1", "bolt_1"])["count"] == 2
assert less_than(2, 3)["result"] and not less_than(4, 3)["result"]
TOOL_CONTRACTS


## 4. Local structured perception proxy

This proxy translates the procedural scene record into evidence and claims. It lets us isolate reasoning mechanics and inject known failures. It does not infer from pixels and must never be reported as foundation-model performance.


In [ ]:
def local_structured_perception_proxy(record: dict[str, Any], injection: str | None = None):
    geometry = scene_geometry(record)
    image_id = record["scene_id"]
    evidence = {
        object_id: Evidence(
            evidence_id=f"{image_id}:{object_id}",
            image_id=image_id,
            object_id=object_id,
            region_xyxy=box,
        )
        for object_id, box in geometry.items()
    }
    confidence = 0.58 if record["low_contrast"] and record["scratch"] else 0.98
    scratch_state = "uncertain" if confidence < 0.70 else ("verified_true" if record["scratch"] else "verified_false")
    scratch_value = record["scratch"]
    scratch_evidence = (evidence["valve_1"].evidence_id,)
    scratch_image = image_id
    if injection == "perception_failure":
        scratch_value = not scratch_value
        scratch_state = "verified_true" if scratch_value else "verified_false"
    elif injection == "missing_fact":
        scratch_value, scratch_state, scratch_evidence = None, "unknown", ()
    elif injection == "wrong_evidence_binding":
        scratch_evidence = (evidence["pipe_1"].evidence_id,)
    elif injection == "unsupported_inference":
        scratch_evidence = ()
    elif injection == "image_binding_failure":
        scratch_image = "wrong-image-id"

    observations = {
        "scratch": Claim("scratch", "observed_attribute", scratch_value, scratch_state, scratch_evidence, ("valve_1",), (scratch_image,), note=f"confidence={confidence:.2f}"),
        "valve_box": Claim("valve_box", "observed_region", geometry["valve_1"], "verified_true", (evidence["valve_1"].evidence_id,), ("valve_1",), (image_id,)),
        "pipe_box": Claim("pipe_box", "observed_region", geometry["pipe_1"], "verified_true", (evidence["pipe_1"].evidence_id,), ("pipe_1",), (image_id,)),
        "bolt_ids": Claim("bolt_ids", "observed_instance_set", tuple(k for k in geometry if k.startswith("bolt_")), "verified_true", tuple(v.evidence_id for k, v in evidence.items() if k.startswith("bolt_")), tuple(k for k in geometry if k.startswith("bolt_")), (image_id,)),
    }
    return observations, evidence, confidence


example_record = records_df.query("split == 'construction'").iloc[1].to_dict()
example_observations, example_evidence, _ = local_structured_perception_proxy(example_record)
pd.DataFrame([asdict(v) for v in example_observations.values()])


## 5. A small DAG executor with four-valued propagation

The graph rejects missing node definitions and cycles. Tool nodes receive only structured predecessors. The terminal policy reviews any decision-relevant `uncertain` or `unknown` fact.

![Three checked facts feed the maintenance rule.](assets/reasoning-dependency-dag.svg)


In [ ]:
@dataclass(frozen=True)
class ReasoningNode:
    node_id: str
    dependencies: tuple[str, ...]
    operation: str


REASONING_NODES = (
    ReasoningNode("left_of", ("valve_box", "pipe_box"), "spatial_relation"),
    ReasoningNode("bolt_count", ("bolt_ids",), "count_objects"),
    ReasoningNode("fewer_than_three", ("bolt_count",), "less_than"),
    ReasoningNode("decision", ("scratch", "left_of", "fewer_than_three"), "maintenance_rule_v1"),
)


def topological_order(nodes: tuple[ReasoningNode, ...], available: set[str]) -> list[ReasoningNode]:
    remaining = {node.node_id: node for node in nodes}
    resolved = set(available)
    ordered = []
    while remaining:
        ready = [node for node in remaining.values() if set(node.dependencies) <= resolved]
        if not ready:
            unresolved = {k: v.dependencies for k, v in remaining.items()}
            raise ValueError(f"cycle or missing dependency: {unresolved}")
        for node in sorted(ready, key=lambda item: item.node_id):
            ordered.append(node); resolved.add(node.node_id); del remaining[node.node_id]
    return ordered


assert [n.node_id for n in topological_order(REASONING_NODES, {"scratch", "valve_box", "pipe_box", "bolt_ids"})][-1] == "decision"
try:
    topological_order((ReasoningNode("a", ("b",), "x"), ReasoningNode("b", ("a",), "x")), set())
    raise AssertionError("cycle should have failed")
except ValueError:
    pass


def derived_state(dependencies: list[Claim]) -> str:
    states = {claim.state for claim in dependencies}
    if "unknown" in states:
        return "unknown"
    if "uncertain" in states:
        return "uncertain"
    return "verified_true"


def execute_reasoning_graph(observations: dict[str, Claim], injection: str | None = None):
    claims = dict(observations)
    events: list[ToolEvent] = []
    order = topological_order(REASONING_NODES, set(observations))
    selected_tool = None
    contradiction_claims = []
    for node in order:
        deps = [claims[name] for name in node.dependencies]
        state = derived_state(deps)
        if node.node_id == "left_of":
            selected_tool = "less_than" if injection == "tool_selection_failure" else "spatial_relation"
            if selected_tool == "spatial_relation":
                event = run_tool(selected_tool, {"subject_box": deps[0].value, "reference_box": deps[1].value}, len(events) + 1)
                value = event.outputs.get("left_of")
            else:
                event = run_tool(selected_tool, {"value": 1, "threshold": 2}, len(events) + 1)
                value = event.outputs.get("result")
            events.append(event)
            if injection == "relation_inversion" and value is not None:
                value = not value
            claims[node.node_id] = Claim(node.node_id, "derived_relation", value, state, deps[0].evidence_ids + deps[1].evidence_ids, ("valve_1", "pipe_1"), deps[0].image_ids, event.tool_event_id)
            if injection == "contradiction":
                contradiction_claims = [
                    {"subject": "valve_1", "relation": "left_of", "object": "pipe_1", "value": True},
                    {"subject": "valve_1", "relation": "right_of", "object": "pipe_1", "value": True},
                ]
        elif node.node_id == "bolt_count":
            selected_ids = list(deps[0].value or [])
            if injection == "tool_input_failure":
                selected_ids.append("valve_1")
            event = run_tool("count_objects", {"object_ids": selected_ids}, len(events) + 1)
            if injection == "tool_execution_failure" and event.status == "ok":
                event.outputs["count"] += 1
            events.append(event)
            claims[node.node_id] = Claim(node.node_id, "derived_quantity", event.outputs.get("count"), state, deps[0].evidence_ids, tuple(selected_ids), deps[0].image_ids, event.tool_event_id)
        elif node.node_id == "fewer_than_three":
            if state in {"unknown", "uncertain"}:
                claims[node.node_id] = Claim(node.node_id, "derived_predicate", None, state)
            else:
                event = run_tool("less_than", {"value": deps[0].value, "threshold": 3}, len(events) + 1)
                if injection == "arithmetic_failure" and event.status == "ok":
                    event.outputs["result"] = not event.outputs["result"]
                events.append(event)
                claims[node.node_id] = Claim(node.node_id, "derived_predicate", event.outputs.get("result"), state, deps[0].evidence_ids, deps[0].object_ids, deps[0].image_ids, event.tool_event_id)
        elif node.node_id == "decision":
            if any(dep.state in {"unknown", "uncertain"} for dep in deps):
                decision = "review_required"
            else:
                decision = "decision_true" if all(bool(dep.value) for dep in deps) else "decision_false"
            if injection == "rule_application_failure" and decision != "review_required":
                decision = "decision_false" if decision == "decision_true" else "decision_true"
            if injection == "abstention_failure" and decision == "review_required":
                decision = "decision_true"
            claims[node.node_id] = Claim(node.node_id, "terminal_decision", decision, "verified_true", tuple(e for dep in deps for e in dep.evidence_ids), tuple(o for dep in deps for o in dep.object_ids), tuple(i for dep in deps for i in dep.image_ids))
    return claims, events, selected_tool, contradiction_claims, [n.node_id for n in order]


## 6. Run one checked question end to end

The tool log distinguishes the system's semantic input selection from exact execution. A tool can execute correctly on the wrong inputs; the checks below score both.


In [ ]:
def run_reasoner(record: dict[str, Any], injection: str | None = None):
    observations, evidence, confidence = local_structured_perception_proxy(record, injection)
    claims, events, selected_tool, contradictions, order = execute_reasoning_graph(observations, injection)
    return {
        "record": record,
        "claims": claims,
        "evidence": evidence,
        "events": events,
        "selected_tool": selected_tool,
        "contradiction_claims": contradictions,
        "order": order,
        "scratch_confidence": confidence,
        "authorization": "none",
    }


example_run = run_reasoner(example_record)
print("question:", "Does the valve require maintenance?")
print("execution order:", example_run["order"])
print("decision:", example_run["claims"]["decision"].value)
pd.DataFrame([asdict(event) for event in example_run["events"]])


In [ ]:
def draw_evidence(record, run):
    image = render_scene(record).copy()
    draw = ImageDraw.Draw(image)
    palette = {"valve_1": (90, 240, 130), "pipe_1": (245, 160, 55)}
    for object_id, item in run["evidence"].items():
        color = palette.get(object_id, (245, 225, 65))
        draw.rectangle(item.region_xyxy, outline=color, width=2)
        draw.text((item.region_xyxy[0], max(0, item.region_xyxy[1] - 10)), object_id, fill=color)
    return image


fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(render_scene(example_record)); axes[0].set_title("Input scene")
axes[1].imshow(draw_evidence(example_record, example_run)); axes[1].set_title("Bound evidence IDs")
for ax in axes: ax.axis("off")
plt.tight_layout()


## 7. Score evidence independently of the answer

For counting, complete evidence means every required unique bolt is represented. For the relation, both subject and reference are required. For the scratch, the named valve region is required.


In [ ]:
def evidence_checks(run) -> pd.DataFrame:
    record = run["record"]
    claims = run["claims"]
    expected = {
        "scratch": {f"{record['scene_id']}:valve_1"},
        "left_of": {f"{record['scene_id']}:valve_1", f"{record['scene_id']}:pipe_1"},
        "bolt_count": {f"{record['scene_id']}:bolt_{i}" for i in range(1, record["bolt_count"] + 1)},
    }
    rows = []
    all_known = {item.evidence_id for item in run["evidence"].values()}
    for claim_id, required in expected.items():
        provided = set(claims[claim_id].evidence_ids)
        overlap = required & provided
        rows.append({
            "claim_id": claim_id,
            "required_count": len(required),
            "provided_count": len(provided),
            "relevant": bool(overlap) if required else True,
            "correct": bool(provided <= all_known and overlap == provided),
            "sufficient": bool(required <= provided),
            "coverage": 1.0 if not required else len(overlap) / len(required),
        })
    return pd.DataFrame(rows)


example_evidence_checks = evidence_checks(example_run)
assert example_evidence_checks["sufficient"].all()
example_evidence_checks


## 8. Development-only review policy

Only scratch confidence needs a threshold in this toy pipeline. We select it using Factory B and freeze it before touching Factory C. This is a teaching policy, not a calibrated production threshold.


In [ ]:
def opaque_answer_proxy(record):
    # Deliberately wrong shortcut: appearance and scratch, ignoring side and bolt count.
    return bool(record["scratch"] and record["color"] == "red")


def apply_scratch_threshold(record, threshold):
    run = run_reasoner(record)
    scratch = run["claims"]["scratch"]
    if run["scratch_confidence"] < threshold:
        scratch.state = "uncertain"
    elif scratch.value is not None:
        scratch.state = "verified_true" if scratch.value else "verified_false"
    claims, events, selected_tool, contradictions, order = execute_reasoning_graph({k: v for k, v in run["claims"].items() if k in {"scratch", "valve_box", "pipe_box", "bolt_ids"}})
    run.update({"claims": claims, "events": events, "selected_tool": selected_tool, "contradiction_claims": contradictions, "order": order})
    return run


def policy_metrics(rows, threshold):
    results = []
    for record in rows:
        run = apply_scratch_threshold(record, threshold)
        decision = run["claims"]["decision"].value
        committed = decision != "review_required"
        predicted = decision == "decision_true"
        correct = committed and predicted == record["decision_truth"]
        unsupported_commit = committed and run["claims"]["scratch"].state in {"uncertain", "unknown"}
        results.append((committed, correct, unsupported_commit))
    committed_n = sum(v[0] for v in results)
    return {
        "threshold": threshold,
        "coverage": committed_n / len(results),
        "selective_accuracy": sum(v[1] for v in results) / max(1, committed_n),
        "unsupported_commit_rate": sum(v[2] for v in results) / len(results),
        "review_rate": 1 - committed_n / len(results),
    }


development_rows = records_df.query("split == 'development'").to_dict(orient="records")
candidate_policy = pd.DataFrame([policy_metrics(development_rows, threshold) for threshold in [0.50, 0.70, 0.90]])
candidate_policy["objective"] = candidate_policy["selective_accuracy"] - 2 * candidate_policy["unsupported_commit_rate"] - 0.10 * candidate_policy["review_rate"]
selected_threshold = float(candidate_policy.sort_values(["objective", "threshold"], ascending=[False, True]).iloc[0]["threshold"])
FROZEN_REVIEW_POLICY = {
    "scratch_confidence_min": selected_threshold,
    "selected_on": "Factory B development only",
    "test_role": "Factory C reporting only; no threshold or rule changes",
    "notice": DEMONSTRATION_THRESHOLD_NOTICE,
}
print(FROZEN_REVIEW_POLICY)
candidate_policy


## 9. Node-level, evidence-level, and final evaluation

Answer-only evaluation collapses perception, binding, tools, and rule behavior. We keep them separate and report the held-out source only after the policy is frozen.


In [ ]:
def evaluate_split(split: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = records_df.query("split == @split").to_dict(orient="records")
    scene_rows, node_rows = [], []
    for record in rows:
        run = apply_scratch_threshold(record, FROZEN_REVIEW_POLICY["scratch_confidence_min"])
        decision = run["claims"]["decision"].value
        committed = decision != "review_required"
        predicted = decision == "decision_true"
        checks = evidence_checks(run)
        expected_left = record["side"] == "left"
        expected_fewer = record["bolt_count"] < 3
        expected_values = {
            "scratch": record["scratch"], "left_of": expected_left,
            "bolt_count": record["bolt_count"], "fewer_than_three": expected_fewer,
        }
        for claim_id, expected in expected_values.items():
            claim = run["claims"][claim_id]
            node_rows.append({"split": split, "scene_id": record["scene_id"], "node": claim_id, "correct": claim.value == expected, "state": claim.state})
        scene_rows.append({
            "split": split,
            "scene_id": record["scene_id"],
            "decision_truth": record["decision_truth"],
            "decision": decision,
            "committed": committed,
            "decision_correct": bool(committed and predicted == record["decision_truth"]),
            "opaque_answer_correct": opaque_answer_proxy(record) == record["decision_truth"],
            "evidence_sufficiency": float(checks["sufficient"].mean()),
            "evidence_coverage": float(checks["coverage"].mean()),
            "image_attribution_correct": all(record["scene_id"] in claim.image_ids for claim in [run["claims"]["scratch"], run["claims"]["left_of"], run["claims"]["bolt_count"]]),
        })
    return pd.DataFrame(scene_rows), pd.DataFrame(node_rows)


dev_scenes, dev_nodes = evaluate_split("development")
test_scenes, test_nodes = evaluate_split("test")  # held-out reporting starts here

def summarize_scenes(frame):
    committed = frame[frame["committed"]]
    return {
        "scenes": len(frame),
        "decision_coverage": frame["committed"].mean(),
        "selective_decision_accuracy": committed["decision_correct"].mean(),
        "review_rate": 1 - frame["committed"].mean(),
        "opaque_answer_accuracy": frame["opaque_answer_correct"].mean(),
        "mean_evidence_sufficiency": frame["evidence_sufficiency"].mean(),
        "mean_evidence_coverage": frame["evidence_coverage"].mean(),
        "image_attribution_accuracy": frame["image_attribution_correct"].mean(),
    }


split_summary = pd.DataFrame({
    "development": summarize_scenes(dev_scenes),
    "held_out_test": summarize_scenes(test_scenes),
}).T
node_summary = pd.concat([dev_nodes, test_nodes]).groupby(["split", "node"]).agg(accuracy=("correct", "mean"), uncertain_or_unknown=("state", lambda s: s.isin(["uncertain", "unknown"]).mean())).reset_index()
display(split_summary)
node_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
split_summary[["selective_decision_accuracy", "opaque_answer_accuracy", "mean_evidence_sufficiency"]].plot.bar(ax=axes[0], ylim=(0, 1.05), title="Separate answer and evidence metrics")
test_nodes.groupby("node")["correct"].mean().sort_values().plot.barh(ax=axes[1], xlim=(0, 1.05), title="Held-out node accuracy")
axes[0].set_ylabel("rate")
plt.tight_layout()


## 10. Relevant and irrelevant counterfactuals

Each variant is created inside the test split and changes exactly one declared fact. The validator prevents an accidental two-factor comparison.

![Relevant changes should flip the answer; color should not.](assets/counterfactual-verification.svg)


In [ ]:
COUNTERFACTUAL_FACTS = ("scratch", "side", "bolt_count", "color")

def changed_facts(a, b):
    return [name for name in COUNTERFACTUAL_FACTS if a[name] != b[name]]


def counterfactual_variant(base, fact, value):
    variant = dict(base)
    variant[fact] = value
    variant["scene_id"] = f"{base['scene_id']}-cf-{fact}"
    variant["group_id"] = base["group_id"]  # remains inside the same test group
    variant["decision_truth"] = bool(variant["scratch"] and variant["side"] == "left" and variant["bolt_count"] < 3)
    assert changed_facts(base, variant) == [fact]
    return variant


base = make_record("Factory C", 101, "test")
base.update({"scratch": True, "side": "left", "bolt_count": 2, "color": "red", "low_contrast": False, "decision_truth": True})
variants = {
    "scratch": counterfactual_variant(base, "scratch", False),
    "side": counterfactual_variant(base, "side", "right"),
    "bolt_count": counterfactual_variant(base, "bolt_count", 4),
    "color": counterfactual_variant(base, "color", "blue"),
}
base_decision = run_reasoner(base)["claims"]["decision"].value
cf_rows = []
for fact, variant in variants.items():
    observed = run_reasoner(variant)["claims"]["decision"].value
    expected_change = variant["decision_truth"] != base["decision_truth"]
    cf_rows.append({
        "changed_fact": fact,
        "rule_relevant": fact in {"scratch", "side", "bolt_count"},
        "expected_answer_change": expected_change,
        "observed_answer_change": observed != base_decision,
        "correct": expected_change == (observed != base_decision),
    })
counterfactual_sensitivity_matrix = pd.DataFrame(cf_rows)
relevant = counterfactual_sensitivity_matrix.query("rule_relevant")
irrelevant = counterfactual_sensitivity_matrix.query("not rule_relevant")
counterfactual_metrics = {
    "relevant_sensitivity": float((relevant["expected_answer_change"] == relevant["observed_answer_change"]).mean()),
    "irrelevant_invariance": float((~irrelevant["observed_answer_change"]).mean()),
}
assert counterfactual_sensitivity_matrix["correct"].all()
print(counterfactual_metrics)
counterfactual_sensitivity_matrix


## 11. Relation inverses and contradiction detection

Agreement across equivalent questions is useful, but repeated agreement is not truth. Here a separate geometry calculation establishes the relation, and a constraint checker detects incompatible claims.


In [ ]:
INVERSES = {"left_of": "right_of", "right_of": "left_of", "above": "below", "below": "above", "inside": "contains", "contains": "inside"}

def contradiction_check(relation_claims):
    contradictions = []
    truths = {(c["subject"], c["relation"], c["object"]) for c in relation_claims if c.get("value") is True}
    exclusive_same_orientation = [("left_of", "right_of"), ("above", "below"), ("inside", "contains")]
    for subject, relation, obj in sorted(truths):
        for first, second in exclusive_same_orientation:
            if relation == first and (subject, second, obj) in truths:
                contradictions.append((subject, first, second, obj))
    return contradictions


geometry = scene_geometry(base)
vp = spatial_relation(geometry["valve_1"], geometry["pipe_1"])
pv = spatial_relation(geometry["pipe_1"], geometry["valve_1"])
inverse_consistency = bool(vp["left_of"] == pv["right_of"] and vp["right_of"] == pv["left_of"])
inconsistent_claims = [
    {"subject": "valve_1", "relation": "left_of", "object": "pipe_1", "value": True},
    {"subject": "valve_1", "relation": "right_of", "object": "pipe_1", "value": True},
]
contradictions = contradiction_check(inconsistent_claims)
assert inverse_consistency and contradictions
pd.DataFrame([{"check": "inverse consistency", "passed": inverse_consistency}, {"check": "injected contradiction detected", "passed": bool(contradictions)}])


## 12. Automated failure attribution

We inject one known defect at a time. Attribution recomputes exact geometry, count, arithmetic, and policy results; it does not read the injection label. Production systems lack perfect truth for every semantic fact, so automatic attribution remains bounded by available annotations and independent checks.

![Failures are assigned to the earliest observable boundary.](assets/reasoning-failure-taxonomy.svg)


In [ ]:
def attribute_failure(run) -> str:
    record, claims, events = run["record"], run["claims"], run["events"]
    expected_image = record["scene_id"]
    scratch = claims["scratch"]
    if scratch.value is None:
        return "missing_fact"
    if scratch.value != record["scratch"]:
        return "perception_failure"
    if scratch.image_ids != (expected_image,):
        return "image_binding_failure"
    expected_scratch_evidence = (f"{expected_image}:valve_1",)
    if scratch.evidence_ids and scratch.evidence_ids != expected_scratch_evidence:
        return "wrong_evidence_binding"
    if not scratch.evidence_ids:
        return "unsupported_inference"
    if run["selected_tool"] != "spatial_relation":
        return "tool_selection_failure"
    expected_left = record["side"] == "left"
    if claims["left_of"].value != expected_left:
        return "relation_inversion"
    true_bolts = tuple(f"bolt_{i}" for i in range(1, record["bolt_count"] + 1))
    if tuple(sorted(claims["bolt_count"].object_ids)) != true_bolts:
        return "tool_input_failure"
    count_event = next(event for event in events if event.name == "count_objects")
    reference_count = count_objects(count_event.inputs["object_ids"])["count"]
    if count_event.outputs.get("count") != reference_count:
        return "tool_execution_failure"
    if claims["fewer_than_three"].value != (claims["bolt_count"].value < 3):
        return "arithmetic_failure"
    if contradiction_check(run["contradiction_claims"]):
        return "contradiction"
    decision = claims["decision"].value
    if claims["scratch"].state in {"uncertain", "unknown"} and decision != "review_required":
        return "abstention_failure"
    expected_decision = "review_required" if claims["scratch"].state in {"uncertain", "unknown"} else ("decision_true" if claims["scratch"].value and claims["left_of"].value and claims["fewer_than_three"].value else "decision_false")
    if decision != expected_decision:
        return "rule_application_failure"
    return "no_failure"


failure_base = dict(base)
failure_base["low_contrast"] = False
injections = [
    "perception_failure", "missing_fact", "wrong_evidence_binding", "relation_inversion",
    "tool_selection_failure", "tool_input_failure", "tool_execution_failure",
    "arithmetic_failure", "rule_application_failure", "contradiction",
    "unsupported_inference", "image_binding_failure",
]
failure_rows = []
for injected in injections:
    run = run_reasoner(failure_base, injected)
    failure_rows.append({"injected": injected, "attributed": attribute_failure(run), "correct_attribution": injected == attribute_failure(run)})

# Abstention failure needs an uncertain decision prerequisite.
uncertain_base = dict(base); uncertain_base["low_contrast"] = True
abstention_run = run_reasoner(uncertain_base, "abstention_failure")
failure_rows.append({"injected": "abstention_failure", "attributed": attribute_failure(abstention_run), "correct_attribution": attribute_failure(abstention_run) == "abstention_failure"})
failure_taxonomy_summary = pd.DataFrame(failure_rows)
assert failure_taxonomy_summary["correct_attribution"].all()
failure_taxonomy_summary


## 13. Tool execution correctness versus input correctness

A correct tool can count a contaminated set. Conversely, correct inputs can reach a faulty implementation or corrupted output. We score these two cases separately.

![Semantic extraction and deterministic execution are separate boundaries.](assets/deterministic-tool-boundary.svg)


In [ ]:
tool_case_rows = []
for injection in [None, "tool_input_failure", "tool_execution_failure"]:
    run = run_reasoner(failure_base, injection)
    event = next(event for event in run["events"] if event.name == "count_objects")
    true_ids = {f"bolt_{i}" for i in range(1, failure_base["bolt_count"] + 1)}
    input_ids = set(event.inputs["object_ids"])
    replay = count_objects(event.inputs["object_ids"])
    tool_case_rows.append({
        "case": injection or "clean",
        "tool_input_correct": input_ids == true_ids,
        "tool_execution_correct": event.outputs == replay,
        "business_count_correct": event.outputs.get("count") == failure_base["bolt_count"],
    })
tool_correctness = pd.DataFrame(tool_case_rows)
tool_correctness


## 14. Multi-image change reasoning and image attribution

Facts are verified within each image before comparison. A color change can be visually real but decision-irrelevant. A source-label swap is a binding failure even if both facts exist somewhere in the pair.

![Before and after facts remain source-bound.](assets/multi-image-binding.svg)


In [ ]:
before = dict(base)
before.update({"scene_id": "panel-200-before", "group_id": "panel-200", "scratch": True, "side": "left", "bolt_count": 2, "color": "red", "decision_truth": True})
after = dict(before)
after.update({"scene_id": "panel-200-after", "scratch": False, "color": "blue", "decision_truth": False})
before_run, after_run = run_reasoner(before), run_reasoner(after)

def compare_images(before_run, after_run):
    before_decision = before_run["claims"]["decision"].value
    after_decision = after_run["claims"]["decision"].value
    attribution_ok = (
        all(before_run["record"]["scene_id"] in c.image_ids for c in [before_run["claims"]["scratch"], before_run["claims"]["left_of"], before_run["claims"]["bolt_count"]])
        and all(after_run["record"]["scene_id"] in c.image_ids for c in [after_run["claims"]["scratch"], after_run["claims"]["left_of"], after_run["claims"]["bolt_count"]])
    )
    visual_changes = [name for name in COUNTERFACTUAL_FACTS if before_run["record"][name] != after_run["record"][name]]
    return {
        "before_decision": before_decision,
        "after_decision": after_decision,
        "became_safer": before_decision == "decision_true" and after_decision == "decision_false",
        "visual_changes": visual_changes,
        "decision_relevant_changes": [name for name in visual_changes if name in {"scratch", "side", "bolt_count"}],
        "image_attribution_correct": attribution_ok,
    }


multi_image_result = compare_images(before_run, after_run)
swapped = run_reasoner(before, "image_binding_failure")
image_swap_detected = attribute_failure(swapped) == "image_binding_failure"
assert multi_image_result["became_safer"] and multi_image_result["image_attribution_correct"] and image_swap_detected
multi_image_result | {"image_swap_detected": image_swap_detected}


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
for ax, record, label in zip(axes, [before, after], ["Before", "After"]):
    ax.imshow(render_scene(record)); ax.set_title(f"{label}: decision={record['decision_truth']}\ncolor={record['color']}, scratch={record['scratch']}"); ax.axis("off")
plt.tight_layout()


## 15. Uncertainty and empty/missing evidence policies

Low scratch confidence is not silently converted to `false`. Because scratch is decision-relevant, the rule returns `review_required`. If the same uncertainty affected only color, the maintenance decision would remain computable.


In [ ]:
uncertain = dict(base); uncertain.update({"scene_id": "uncertain-scratch", "low_contrast": True, "scratch": True})
uncertain_run = run_reasoner(uncertain)
missing_run = run_reasoner(failure_base, "missing_fact")
state_policy_examples = pd.DataFrame([
    {"case": "verified inputs", "scratch_state": run_reasoner(failure_base)["claims"]["scratch"].state, "decision": run_reasoner(failure_base)["claims"]["decision"].value},
    {"case": "uncertain scratch", "scratch_state": uncertain_run["claims"]["scratch"].state, "decision": uncertain_run["claims"]["decision"].value},
    {"case": "missing scratch", "scratch_state": missing_run["claims"]["scratch"].state, "decision": missing_run["claims"]["decision"].value},
])
assert set(state_policy_examples.query("case != 'verified inputs'")["decision"]) == {"review_required"}
state_policy_examples


## 16. Model-based verification is a supplementary signal

This lab does not download a judge. A model judge could triage free-form evidence, but it would need a pinned model, processor, prompt, decoding policy, artifact hashes, adjudicated agreement study, and a disagreement route. Exact geometry and policy replay should not be replaced with a generative vote.


In [ ]:
OPTIONAL_MODEL_VERIFIER = {
    "enabled": False,
    "reason": "credential-free deterministic course path; no model-as-ground-truth claim",
    "requirements_if_enabled": [
        "immutable model and processor revisions",
        "artifact hashes and approved licenses",
        "frozen verifier rubric and decoding policy",
        "agreement and disagreement analysis against expert adjudication",
        "tenant isolation, retention, and prompt-injection controls",
    ],
    "authoritative_for_deterministic_checks": False,
}
OPTIONAL_MODEL_VERIFIER


## 17. Stage latency and event observability

These timings describe tiny local Python functions on this notebook runtime. They are not model-serving, network, concurrency, or target-hardware numbers. We time individual reasoning runs, then report a distribution rather than one measurement.


In [ ]:
latencies = []
latency_rows = records_df.query("split == 'test'").to_dict(orient="records")
for _ in range(20):
    for record in latency_rows:
        started = time.perf_counter_ns(); run = apply_scratch_threshold(record, selected_threshold); elapsed = (time.perf_counter_ns() - started) / 1e6
        latencies.append(elapsed)
latency_summary = {
    "samples": len(latencies),
    "median_ms": float(np.median(latencies)),
    "p90_ms": float(np.percentile(latencies, 90)),
    "p95_ms": float(np.percentile(latencies, 95)),
    "iqr_ms": float(np.percentile(latencies, 75) - np.percentile(latencies, 25)),
    "scope": "individual local structured reasoning runs; excludes model inference and network",
}
latency_summary


## 18. Enterprise evidence artifact

The artifact separates locally measured evidence, optional downloaded-model observations, and unresolved production assumptions. It grants no action authority.

![The production architecture keeps reasoning, review, and authorization separate.](assets/enterprise-reasoning-architecture.svg)


In [ ]:
runtime_manifest = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "pillow": Image.__version__, "matplotlib": matplotlib.__version__, "seed": SEED,
}
evidence_artifact = {
    "course": "Intermediate 02 — Multimodal Reasoning & Verification",
    "artifact_type": "advisory_reasoning_evidence",
    "authorization": "none",
    "perception_engine": {"engine": LOCAL_PERCEPTION_LABEL, "foundation_model": False, "pixel_inference": False},
    "split_contract": SOURCE_CONTRACT | {"test_role": "Factory C reporting only; no threshold or rule changes"},
    "locally_measured_evidence": {
        "frozen_review_policy": FROZEN_REVIEW_POLICY,
        "development_summary": summarize_scenes(dev_scenes),
        "held_out_test_summary": summarize_scenes(test_scenes),
        "node_summary": node_summary.to_dict(orient="records"),
        "counterfactual_sensitivity_matrix": counterfactual_sensitivity_matrix.to_dict(orient="records"),
        "counterfactual_metrics": counterfactual_metrics,
        "failure_taxonomy_summary": failure_taxonomy_summary.to_dict(orient="records"),
        "tool_correctness": tool_correctness.to_dict(orient="records"),
        "inverse_consistency": inverse_consistency,
        "contradictions_detected": len(contradictions),
        "multi_image_result": multi_image_result,
        "latency": latency_summary,
        "tool_contracts": {k: asdict(v) for k, v in TOOL_CONTRACTS.items()},
    },
    "optional_downloaded_model_observations": OPTIONAL_MODEL_VERIFIER,
    "demonstration_thresholds_for_this_notebook_runtime_only": {"scratch_confidence_min": selected_threshold},
    "unresolved_production_assumptions": [
        "representative governed images and rare/failure slices",
        "validated perception model, processor, prompt, artifacts, and license",
        "expert-reviewed semantic ground truth and inter-reviewer agreement",
        "calibrated uncertainty and human-review capacity",
        "target-hardware latency, memory, concurrency, availability, and cost",
        "prompt-injection, OCR-text, privacy, retention, tenant, and regional controls",
        "rule ownership, change approval, rollback, and monitoring",
        "separate authenticated authorization for any real maintenance action",
    ],
    "runtime": runtime_manifest,
}
artifact_dir = Path("artifacts"); artifact_dir.mkdir(exist_ok=True)
artifact_path = artifact_dir / "intermediate-02-multimodal-reasoning-evidence.json"
artifact_path.write_text(json.dumps(evidence_artifact, indent=2) + "\n", encoding="utf-8")
print(artifact_path)
print(json.dumps({"authorization": evidence_artifact["authorization"], "test": evidence_artifact["locally_measured_evidence"]["held_out_test_summary"]}, indent=2))


## 19. Final assertions

These checks protect the course contract: clean evidence is sufficient, tool input and execution failures remain distinguishable, relevant/irrelevant counterfactuals behave as specified, image swaps are detected, uncertain decision facts route to review, and Factory C remains reporting-only.


In [ ]:
assert FROZEN_REVIEW_POLICY["selected_on"] == "Factory B development only"
assert "reporting only" in FROZEN_REVIEW_POLICY["test_role"]
assert failure_taxonomy_summary["correct_attribution"].all()
assert counterfactual_metrics == {"relevant_sensitivity": 1.0, "irrelevant_invariance": 1.0}
assert tool_correctness.set_index("case").loc["tool_input_failure", "tool_execution_correct"]
assert not tool_correctness.set_index("case").loc["tool_input_failure", "tool_input_correct"]
assert tool_correctness.set_index("case").loc["tool_execution_failure", "tool_input_correct"]
assert not tool_correctness.set_index("case").loc["tool_execution_failure", "tool_execution_correct"]
assert image_swap_detected
assert state_policy_examples.query("case == 'uncertain scratch'").iloc[0]["decision"] == "review_required"
assert evidence_artifact["authorization"] == "none"
assert evidence_artifact["perception_engine"] == {"engine": LOCAL_PERCEPTION_LABEL, "foundation_model": False, "pixel_inference": False}
assert artifact_path.exists()
print("All Intermediate 02 reasoning, evidence, tool, counterfactual, attribution, and governance checks passed.")


## 20. Practice and checkpoint

Try these in order:

1. Change `left_of` from center-based to edge-based and version the tool contract.
2. Add an occluded bolt state; decide whether it is verified, uncertain, or ignored before coding.
3. Inject duplicate bolt evidence and compare entity coverage with raw evidence count.
4. Add `inside ↔ contains` with an explicit boundary-touching policy.
5. Add a counterfactual validator that rejects changes to two causal facts.
6. Add a simulated reviewer outcome and measure disagreement without treating the reviewer as infallible.

You should now be able to explain, without code:

- why an answer can be right with wrong evidence;
- why a relation needs both subject and reference evidence;
- why correct tool execution can still produce a wrong business conclusion;
- why `unknown` is not `false`;
- why relevant sensitivity and irrelevant invariance are separate metrics;
- why self-consistency is not truth; and
- why a reasoning result is not authorization for an agent to act.

Next: **Intermediate 03 — Document Intelligence**, where the same evidence contract must survive pages, reading order, OCR, tables, charts, and long-document provenance.
